# ToTTI B+C — fine-tune opus-mt-tc-big-en-it (distilled + human anchor)

Runtime: **T4 GPU** is enough (0.2B params). Expected cost ~2-4 units. Data uploaded: the
AMI/ICSI-derived distilled pairs (CC BY 4.0 derivatives), the Tatoeba anchor (CC BY 2.0 FR)
and FLORES-200 (CC BY-SA 4.0) — the private test set never comes here; the accepting scores
run at home. The gate is pre-registered in config-bc.yaml.

Steps: run all, upload `totti-bc-bundle.zip` when asked, authorize Drive, and at the end both
zips are PARKED ON DRIVE (the tab download died once to the idle timeout — Drive survives).

In [ ]:
gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(gpu_info)
assert gpu_info and "failed" not in gpu_info[0], "Runtime -> Change runtime type -> T4 GPU"

In [ ]:
!pip -q install "transformers<5" sacrebleu ctranslate2 sentencepiece accelerate protobuf

In [ ]:
# Upload totti-bc-bundle.zip (config-bc.yaml, train.py, convert.sh, data/).
from google.colab import files

uploaded = files.upload()
!unzip -o -q totti-bc-bundle.zip -d /content/totti
!wc -l /content/totti/data/*

In [ ]:
# Training with the config echo first (so the log proves what runs), then the
# provisional FLORES verdict. ~10-25 min on a T4.
!cd /content/totti && python train.py --config config-bc.yaml --out out

In [ ]:
# CT2 int8 conversion of the best checkpoint.
!cd /content/totti && bash convert.sh out/best out/opus-mt-tc-big-totti-ct2
!du -sh /content/totti/out/opus-mt-tc-big-totti-ct2

In [ ]:
# Park BOTH results on Drive first (never trust the tab with gigabytes):
# the CT2 model for the app, and the fp16 best checkpoint for the alpha-sweep at home.
!cd /content/totti/out && zip -r -q /content/tc-big-ct2.zip opus-mt-tc-big-totti-ct2
!cd /content/totti/out && zip -r -q /content/tc-big-best-fp16.zip best
from google.colab import drive

drive.mount("/content/drive")
!cp /content/tc-big-ct2.zip /content/tc-big-best-fp16.zip /content/drive/MyDrive/
!ls -lah /content/drive/MyDrive/tc-big-*.zip
print("fatto: scaricali da drive.google.com, poi Runtime -> Disconnetti ed elimina runtime")

## At home

Download both zips from Drive, then tell Claude: the alpha-sweep (student x tc-big base) and
the accepting scores against the pre-registered gate run on the Mac. Delete the zips from
Drive afterwards to free the space.